# Exercise 18: Cost and Token Tracking

Extract usage metadata from every Responses API call, calculate per-call costs
(including cached-input discounts), and build a **session cost summary**
with monthly extrapolation.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

## Pricing table (per 1M tokens)

In [ ]:
PRICING = {
    "gpt-4.1-nano-2025-04-14": {"input": 0.10, "output": 0.40, "cached_input": 0.025},
    "gpt-4.1-mini-2025-04-14": {"input": 0.40, "output": 1.60, "cached_input": 0.10},
    "gpt-4.1-2025-04-14":      {"input": 2.00, "output": 8.00, "cached_input": 0.50},
    # Aliases
    "gpt-4.1-nano": {"input": 0.10, "output": 0.40, "cached_input": 0.025},
    "gpt-4.1-mini": {"input": 0.40, "output": 1.60, "cached_input": 0.10},
    "gpt-4.1":      {"input": 2.00, "output": 8.00, "cached_input": 0.50},
}

## Cost calculation helpers

In [ ]:
def calculate_cost(response):
    """Extract usage and calculate cost from a Responses API response."""
    usage = response.usage
    model = response.model

    # Get pricing (try exact model name, then strip version)
    prices = PRICING.get(model, PRICING.get(model.split("-2025")[0], {}))
    if not prices:
        return {"error": f"No pricing for {model}"}

    # Token breakdown
    input_tokens = usage.input_tokens
    output_tokens = usage.output_tokens
    cached_tokens = usage.input_tokens_details.cached_tokens if usage.input_tokens_details else 0
    non_cached_input = input_tokens - cached_tokens

    # Cost calculation
    input_cost = (non_cached_input / 1_000_000) * prices["input"]
    cached_cost = (cached_tokens / 1_000_000) * prices.get("cached_input", prices["input"])
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    total_cost = input_cost + cached_cost + output_cost

    return {
        "model": model,
        "input_tokens": input_tokens,
        "cached_tokens": cached_tokens,
        "non_cached_input": non_cached_input,
        "output_tokens": output_tokens,
        "total_tokens": usage.total_tokens,
        "input_cost": input_cost,
        "cached_cost": cached_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


def print_cost_report(cost):
    """Pretty-print a cost breakdown."""
    print(f"  Model:            {cost['model']}")
    print(f"  Input tokens:     {cost['input_tokens']:>8} (cached: {cost['cached_tokens']})")
    print(f"  Output tokens:    {cost['output_tokens']:>8}")
    print(f"  Total tokens:     {cost['total_tokens']:>8}")
    print(f"  Input cost:       ${cost['input_cost']:.6f}")
    print(f"  Cached cost:      ${cost['cached_cost']:.6f}")
    print(f"  Output cost:      ${cost['output_cost']:.6f}")
    print(f"  TOTAL COST:       ${cost['total_cost']:.6f}")

## Call 1: Simple question (gpt-4.1-mini)

In [ ]:
costs = []

print("--- Call 1: Simple question (gpt-4.1-mini) ---")
r1 = client.responses.create(
    model="gpt-4.1-mini",
    input="What is the Responses API?",
)
c1 = calculate_cost(r1)
costs.append(c1)
print_cost_report(c1)

## Call 2: Longer generation (gpt-4.1-mini)

In [ ]:
print("--- Call 2: Longer generation (gpt-4.1-mini) ---")
r2 = client.responses.create(
    model="gpt-4.1-mini",
    input="Write a detailed 5-step implementation plan for deploying a RAG system in an enterprise environment.",
)
c2 = calculate_cost(r2)
costs.append(c2)
print_cost_report(c2)

## Call 3: With web_search tool (gpt-4.1-mini)

In [ ]:
print("--- Call 3: With web_search tool (gpt-4.1-mini) ---")
r3 = client.responses.create(
    model="gpt-4.1-mini",
    tools=[{"type": "web_search"}],
    input="What are the latest updates to OpenAI's enterprise offerings?",
)
c3 = calculate_cost(r3)
costs.append(c3)
print_cost_report(c3)

## Call 4: Same prompt on premium model (gpt-4.1)

In [ ]:
print("--- Call 4: Same question on gpt-4.1 ---")
r4 = client.responses.create(
    model="gpt-4.1",
    input="Write a detailed 5-step implementation plan for deploying a RAG system in an enterprise environment.",
)
c4 = calculate_cost(r4)
costs.append(c4)
print_cost_report(c4)

## Session cost summary

In [ ]:
print("=" * 60)
print("SESSION COST SUMMARY")
print("=" * 60)

total_session = sum(c["total_cost"] for c in costs)
total_tokens = sum(c["total_tokens"] for c in costs)

print(f"\n{'Call':<35} {'Tokens':>8} {'Cost':>12}")
print("-" * 60)
for i, c in enumerate(costs, 1):
    print(f"Call {i} ({c['model'][:20]}){'':<10} {c['total_tokens']:>8} ${c['total_cost']:>10.6f}")
print("-" * 60)
print(f"{'TOTAL':<35} {total_tokens:>8} ${total_session:>10.6f}")

## Monthly cost extrapolation

In [ ]:
calls_per_day = 10000
daily_cost = (total_session / len(costs)) * calls_per_day
monthly_cost = daily_cost * 30
print(f"Avg cost per call:  ${total_session / len(costs):.6f}")
print(f"At {calls_per_day:,} calls/day: ${daily_cost:.2f}/day, ${monthly_cost:.2f}/month")

## Mini vs full model comparison

In [ ]:
print("Call 2 (mini) vs Call 4 (4.1) with same prompt:")
print(f"  mini: {c2['total_tokens']} tokens, ${c2['total_cost']:.6f}")
print(f"  4.1:  {c4['total_tokens']} tokens, ${c4['total_cost']:.6f}")
if c4['total_cost'] > 0:
    print(f"  4.1 is {c4['total_cost']/c2['total_cost']:.1f}x more expensive")

## Interview one-liner

> Every Responses API call returns `usage.input_tokens`, `usage.output_tokens`, and `input_tokens_details.cached_tokens` -- multiply each bucket by its per-million rate, and you have **exact per-request cost tracking** with no external metering needed.